In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm, trange

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader
import torch.nn.functional as F
from torchvision.transforms import ToTensor
from torchvision import models

np.random.seed(0)
torch.manual_seed(0)

In [ ]:
from keras.preprocessing.image import ImageDataGenerator

BATCH_SIZE=32

SEED=1345

train_datagen=ImageDataGenerator(rescale=1./255,
                                zoom_range = 0.2)

validation_datagen=ImageDataGenerator(rescale=1./255)
test_datagen=ImageDataGenerator(rescale=1./255)


#Defining directories for train,validation,test 
train_dir = 'Splitted2D/train'
validation_dir = 'Splitted2D/val'
test_dir = 'Splitted2D/test'


#Defining generatores for train,validation,test

train_generator=train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    shuffle=True,
    batch_size= 499,
    class_mode ='categorical',
)

validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=(224, 224),
    shuffle=True,
    batch_size= 62,
    class_mode ='categorical',
)

# Define generator for test set using flow_from_directory
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    shuffle=True,
    batch_size = 66,
    class_mode ='categorical',
)

In [ ]:
x_train, y_train = next(train_generator)
x_train = x_train

x_val, y_val = next(validation_generator)
x_val = x_val

x_test, y_test = next(test_generator)
x_test = x_test

In [ ]:
# data augmentation pipeline
import torchvision.transforms as transforms
from torch.utils.data import TensorDataset

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(45),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

augmented_train = torch.stack([transform(s[:, :, 0]) for s in x_train])
augmented_val = torch.stack([transform(s[:, :, 0]) for s in x_val])
augmented_test = torch.stack([transform(s[:, :, 0]) for s in x_test])
y_train_tensor = torch.tensor([np.argmax(l) for l in y_train])
y_val_tensor = torch.tensor([np.argmax(l) for l in y_val])
y_test_tensor = torch.tensor([np.argmax(l) for l in y_test])

train_set = TensorDataset(augmented_train, y_train_tensor)
val_set = TensorDataset(augmented_val, y_val_tensor)
test_set = TensorDataset(augmented_test, y_test_tensor)

batch_size = 32

train_loader = DataLoader(train_set, shuffle = False, batch_size = batch_size)
val_loader = DataLoader(val_set, shuffle = False, batch_size = batch_size)
test_loader = DataLoader(test_set, shuffle = False, batch_size = batch_size)

In [ ]:
# dhw = (1, 256, 256), hidden_d = 640, 2048, 2560 (first try 640)
class ViT(nn.Module):
    def __init__(self, dhw, patch_size, hidden_d, n_blocks, n_heads, out_d):
        # Super constructor
        super(ViT, self).__init__()
         
        # Attributes
        self.patch_size = patch_size # this is the (d, h, w) for patches (patch size)
        self.hidden_d = hidden_d
        
        assert dhw[1] % patch_size == 0, "Image size not divisible by patch size"
        
        self.n_patches = (dhw[1] // patch_size)
        
        # still need to flatten these patches
        
        # 1) Linear mapping
        self.input_d = int(patch_size * patch_size) # input_d = 256
        self.linear_mapper = nn.Linear(self.input_d, self.hidden_d) # hidden_d = 512
        
        # 2) Learnable classication token
        self.class_token = nn.Parameter(torch.rand(1, self.hidden_d))
        
        # 3) Positional embeddings --> could try different positional embeddings in the future
        self.register_buffer('positional_embeddings', get_positional_embeddings(self.n_patches ** 2 + 1, hidden_d), 
                             persistent=False)
        
        # 4) Transformer Encoder Blocks
        self.blocks = nn.ModuleList([ViTBlock(hidden_d, n_heads) for _ in range(n_blocks)]) 
        # n_blocks should be 2(thats what it usually is) and n_heads is still up to fine tuning
        
        # Classification MLP
        self.mlp = nn.Sequential(
            nn.LayerNorm(hidden_d),
            nn.Identity(),
            nn.Dropout(p = 0.2, inplace = False),
            nn.Linear(self.hidden_d, out_d),
            #nn.Conv2d(self.hidden_d, out_d, kernel_size = 3, stride = 1, padding = 1),
            #nn.LeakyReLU(0.1),
            nn.Softmax(dim = -1),
            nn.LayerNorm(out_d)
        )
        
        self.init_weights()
    
    def init_weights(self):
        for module in self.mlp.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight)
                nn.init.normal_(module.bias)
    
    
    def forward(self, images):
        n, c, h, w = np.shape(images)
        
        # FIX PATCHIFY (NOT DONE)
        patches = patchify(images, self.n_patches).to(self.positional_embeddings.device)
        
        # linear vectorization/projection (for the first instance its technically reduction or smth)
        tokens = self.linear_mapper(patches)
        
        # Adding classification token
        tokens = torch.cat((self.class_token.expand(n, 1, -1), tokens), dim=1)
        
        # Adding pos embedding
        out = tokens + self.positional_embeddings.repeat(n, 1, 1)
        
        # Transformer blocks
        for block in self.blocks:
            out = block(out)
        
        # Getting classification token only
        out = out[:, 0]
        #print(np.shape(self.mlp(out)))
        return self.mlp(out) # map to output dimension, output category distribution 

In [ ]:
def get_positional_embeddings(sequence_length, d):
    result = torch.ones(sequence_length, d)
    for i in range(sequence_length):
        for j in range(d):
            result[i][j] = np.sin(i / (10000 ** (j / d))) if j % 2 == 0 else np.cos(i / (10000 ** ((j - 1) / d)))
    return result

In [ ]:
class ViTBlock(nn.Module):
    def __init__(self, hidden_d, n_heads, mlp_ratio=4):
        super(ViTBlock, self).__init__()
        self.hidden_d = hidden_d
        self.n_heads = n_heads

        self.norm1 = nn.LayerNorm(hidden_d)
        self.mhsa = MSA(hidden_d, n_heads)
        self.norm2 = nn.LayerNorm(hidden_d)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_d, mlp_ratio * hidden_d),
            #nn.Conv2d(hidden_d, mlp_ratio * hidden_d, kernel_size = 3, stride = 1, padding = 1),
            #nn.LeakyReLU(0.1),
            nn.GELU(),
            nn.Dropout(p=0.2, inplace=False),
            nn.Identity(),
            nn.Linear(mlp_ratio * hidden_d, hidden_d),
            nn.Dropout(p=0.2, inplace=False),
            nn.LayerNorm(hidden_d)
            #nn.Conv2d(mlp_ratio * hidden_d, hidden_d, kernel_size = 3, stride = 1, padding = 1),
            #nn.LeakyReLU(0.1)
        )
        
        #self.norm3 = nn.Sequential(
        #    nn.LayerNorm(hidden_d),
        #    nn.Identity()
        #)
        
        self.init_weights()
    
    def init_weights(self):
        for module in self.mlp.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight)
                nn.init.normal_(module.bias)

    def forward(self, x):
        out = x + self.mhsa(self.norm1(x))
        out = out + self.mlp(self.norm2(out))
        #out += self.norm3(out)
        return out


In [ ]:
class MSA(nn.Module):
    def __init__(self, d, n_heads):
        super(MSA, self).__init__()
        self.d = d # hidden_d --> dimensionality of each token
        self.n_heads = n_heads # compares all tokens but only in a specific set of their values in the hidden_d dim

        assert d % n_heads == 0, f"Can't divide dimension {d} into {n_heads} heads"
        
        # find a way to add regularization and normalization layers to the mappings --> this is most likely
        #  whats causing the vanishing gradient because most of the parameters are probably here
        d_head = int(d / n_heads) # elements per head
        self.q_mappings = nn.ModuleList([nn.Linear(d_head, d_head) for _ in range(self.n_heads)])
        self.k_mappings = nn.ModuleList([nn.Linear(d_head, d_head) for _ in range(self.n_heads)])
        self.v_mappings = nn.ModuleList([nn.Linear(d_head, d_head) for _ in range(self.n_heads)])
        #self.msa = nn.MultiheadAttention(embed_dim = d_head, num_heads = n_heads, dropout = 0.4)
        self.d_head = d_head
        self.softmax = nn.Softmax(dim=-1)
        
        self.dropout = nn.Dropout(p = 0.5, inplace = False)
        
    def forward(self, sequences):
        # Sequences(token sequences) has shape (N, seq_length, token_dim)
        # We go into shape    (N, n_heads, seq_length, token_dim / n_heads)
        # And come back to    (N, seq_length, item_dim)  (through concatenation)
        result = []
        for sequence in sequences:
            seq_result = []
            for head in range(self.n_heads):
                q_mapping = self.q_mappings[head]
                k_mapping = self.k_mappings[head]
                v_mapping = self.v_mappings[head]
                

                seq = sequence[:, head * self.d_head: (head + 1) * self.d_head] # accesses the tokens for each head
                q, k, v = self.dropout(q_mapping(seq)), self.dropout(k_mapping(seq)), v_mapping(seq)

                attention = self.softmax(q @ k.T / (self.d_head ** 0.5))
                #attention, _ = self.msa(seq, seq, seq)
                #seq_result.append(attention)
                seq_result.append(attention @ v)
            result.append(torch.hstack(seq_result))
        return torch.cat([torch.unsqueeze(r, dim=0) for r in result])   

In [ ]:

def patchify(images, n_patches):
    n, c, h, w = images.shape

    patches = torch.zeros(n, n_patches ** 2, h * w // n_patches ** 2)
    patch_size = h // n_patches

    for idx, image in enumerate(images):
        for i in range(n_patches):
            for j in range(n_patches):
                patch = image[:, i * patch_size: (i + 1) * patch_size, j * patch_size: (j + 1) * patch_size]
                patches[idx, i * n_patches + j] = patch.flatten()
    return patches

In [ ]:
print(x_train.shape)

In [ ]:
for param in model.parameters():
    print(param)

In [ ]:
dhw = (1, 224, 224)
#print(np.shape(x_train))
samples, c, h, w = np.shape(x_train)
patch_size = 16

N_EPOCHS = 15
LR = 0.005
batch_size = 32 # can increase in the future
device = torch.device('cpu')

model = ViT(dhw, patch_size, hidden_d = 512, n_blocks = 2, n_heads = 8, out_d = 4).to(device)

optimizer = Adam(model.parameters(), lr = LR)
criterion = CrossEntropyLoss()

for epoch in trange(N_EPOCHS, desc = "TRAINING"):
    correct, total = 0, 0
    train_loss = 0.0
    for batch in tqdm(train_loader, desc = f"EPOCH {epoch + 1} IN TRAINING", leave = False):
        x, y = batch
        
        x, y = x.to(device), y.to(device)
        y_hat = model(x.float())
        
        #print(y_hat.shape, y.shape)
        loss = criterion(y_hat, y)
        
        train_loss += loss.detach().cpu().item() / len(train_loader)
        
        optimizer.zero_grad()
#         for name, param in model.named_parameters():
#             if param.grad is not None:
#                 gradient_norm = torch.norm(param.grad)
#                 print(gradient_norm)
        
        loss.backward()
        optimizer.step()
    
    for batch in tqdm(val_loader, desc = 'VALIDATION'):
        x, y = batch
        x, y = x.to(device), y.to(device)
        
        y_hat = model(x.float())
        
        correct += torch.sum(torch.argmax(y_hat, dim = 1) == y).detach().cpu().item()
        print(torch.argmax(y_hat, dim = 1))
        total += len(x)
    
    print(f"EPOCH {epoch + 1} / {N_EPOCHS} loss: {train_loss:.2f}")
    print(f"Validation accuracy: {correct / total * 100:.2f}%")

with torch.no_grad():
    correct, total = 0, 0
    test_loss = 0.0
    for batch in tqdm(test_loader, desc = "TESTING"):
        x, y = batch

        x, y = x.to(device), y.to(device)
        
        y_hat = model(x.float())

        loss = criterion(y_hat, y)

        test_loss += loss.detach().cpu().item() / len(test_loader)

        correct += torch.sum(torch.argmax(y_hat, dim=1) == y).detach().cpu().item()
        total += len(x)
    print(f"Test loss: {test_loss:.2f}")
    print(f"Test accuracy: {correct / total * 100:.2f}%")
    